Ce fichier test RandomForest comme seuil du mouvement. Chaque modèle prédit pour 1 sec un "mouvement" ou un "non mouvement", puis ces résultats sont comparés aux annotations disponnibles dans "C:/Users/roman/Documents/BEaCHILD/X_et_Y" (correspondent aux annotations RCT1 de CAP). Les modèles sont évalués par le score F1, le score d'accuracy et l'analyse de la matrice de confusion. L'idée est que le RF est entrainé en leave one out sur tous les fichiers (récupère les données des n fichiers (tous) puis l'entraine sur l'ensemble des n - 1 fichiers et le test sur le dernier fichier. Puis l'entrainement est répété pour laisser de coté une fois à leur tour chaque fichier (pour le test)).

Les étapes réalisées sont les suivantes:
1. Lecture des fichiers .csv (données des capteurs), conversion dans la bonne unité et transformation en AC
2. Récuperation des annotations
3. Association des annotations des fichiers excel aux valeurs des fichiers .csv (fait dans 1. et 2.)
4. Entrainement de Random Forest. 
5. Analyser des résultats (matrices de confusion, accuracy score)

Plusieurs tests ont été effectués pour pallier le déséquilibre des classes 'mouvement' et 'non mouvement':
- équilibrage des données (séléction d'une partie (aléatoire) des données de la classe majoritaire),
- utilisation de BaggingClassifier à la place de random forest classique
- utilisation de BalancedRandomForestClassifier

In [7]:
import os
import numpy as np
import pandas as pd
import openpyxl
import joblib
from agcounts_filter import convert_AC
import matplotlib.pyplot as plt
from sklearn.model_selection import LeaveOneOut
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import f1_score

In [8]:
folder_path = "C:/Users/roman/Documents/BEaCHILD/X_et_Y" 
folder = os.listdir(folder_path)

#////////////////////////////// Attention : le fichier LW est défini comme le dominant ////////////////////////////////

# Initialisation des listes regroupant les données de capteurs de tous les fichiers (un fichier = une liste)
all_X_dom = []
all_X_non_dom = []

# Initialisation des listes regroupant les annotations de tous les fichiers (un fichier = une liste)
all_Y_dom = []
all_Y_non_dom = []

len_per_file_dom = []
len_per_file_non_dom = []
previous_len_dom = 0
previous_len_non_dom = 0

#Parcourir les fichiers CSV
for file in folder: # //////////////////////// Ajuster les indices pour choisir le fichier à analyser
    print(f"On est dans le fichier : {file}")
    
    # Extension du fichier
    extension = os.path.splitext(file)[1] 

    # Pour les cas ou il y a des annotations avant le start
    allow_start_dom = None
    allow_start_non_dom = None

    """********************************************************************** 1. Lecture des fichiers csv ******************************************************************************"""
    if extension == ".csv":

        # Visualisation des données des capteurs
        data_file = pd.read_csv(folder_path + "/" + file, header = 5, names = ["Timestamp","Gyro X","Gyro Y","Gyro Z","Accelerometer X","Accelerometer Y","Accelerometer Z","Event","Quat W","Quat X","Quat Y","Quat Z","None"])    

        # Conversion en AC 
        dom_AC = convert_AC(folder_path + "/" + file)

        # Enregistrement des données des capteurs dans les listes qui seront donnnées au classificateur
        if file[-11:-4] == "non_dom":

            # Initialisation des listes contenant les données des capteurs
            X_non_dom = []

            list_dom_AC = dom_AC["AC"].tolist() # Conversion du type pd.serie en type list
            for ac in list_dom_AC :
                X_non_dom.append(ac)
            

        elif file[-7:-4] == "dom":

            # Initialisation des listes contenant les données des capteurs
            X_dom = []

            list_dom_AC = dom_AC["AC"].tolist()
            for ac in list_dom_AC:
                X_dom.append(ac)

    """********************************************************** 2. Enregistrer les annotations ***************************************************************************"""

    if extension == ".xlsx":

        # Initialisation des listes contenant les annotations vidéos
        Y_non_dom = [] 
        Y_dom = []

        my_wb = openpyxl.load_workbook(folder_path + "/" + file) 
        my_sheet = my_wb.active

        # Initialisation des décalages : sert pour éviter le décalage du aux arrondis des annotations 
        decalage_dom = 0
        decalage_non_dom = 0

        for label in my_sheet["H"]: 

            """*********************************************** Récupération du début de Y ****************************************************"""
            # dom
            # Synchronisation des données X des capteurs et des annotations Y
            if label.value == "Start_LW" :
                """start_dom = int(round(float(my_sheet.cell(label.row,13).value))) - 1
                # Décalage pour synchroniser X et Y
                for decalage in range(start_dom):
                    Y_dom.append(None)"""
                previous_row_dom = label.row
                # Pour les cas ou il y a des annotations avant le start 
                allow_start_dom = True

            # non_dom 
            # Synchronisation des données X des capteurs et des annotations Y
            if label.value == "Start_RW" :
                """BrokenPipeError"""
                previous_row_non_dom = label.row
                # Pour les cas ou il y a des annotations avant le start 
                allow_start_non_dom = True


            """****************************************************** Récupération du contenu de Y ****************************************"""

            if label.value[0:2] == "LW" and allow_start_dom:

                # Récupération de l'index de la ligne
                row = label.row

                # Synchronisation si le start de l'annotation ne correspond pas au stop de l'annotation précédentes
                if my_sheet.cell(row, 12).value != my_sheet.cell(previous_row_dom, 13).value:
                    difference = round(float(my_sheet.cell(row, 12).value)) - round(float(my_sheet.cell(previous_row_dom , 13).value)) 
                    for diff in range(0,difference):
                        Y_dom.append(None) #Y_dom.append(None) 
                    
                # Synchronisation des données des capteurs avec les annotations
                nb_repetitions = int(round(float(my_sheet.cell(label.row, 13).value))) - int(round(float(my_sheet.cell(label.row, 12).value))) 
                for nb_sec in range(nb_repetitions): 
                    if X_dom and (len(Y_dom) == len(X_dom)) :
                        break
                    if label.value[3:13] == "sédentaire" :
                        Y_dom.append("non mouvement")
                    elif label.value[-2:] == "GA" or label.value[-2:] == "FA":
                        Y_dom.append("mouvement")
                    elif label.value[3:12] == "non noté":
                        Y_dom.append(None) 

                #Au prochain tour, l'indice de la ligne actuelle sera l'index de la ligne précédente
                previous_row_dom = label.row


            elif label.value[0:2] == "RW" and allow_start_non_dom:
                
                # Récupération de l'index de la ligne
                row = label.row

                # Synchronisation si le start de l'annotation ne correspond pas au stop de l'annotation précédentes
                if my_sheet.cell(row, 12).value != my_sheet.cell(previous_row_non_dom, 13).value:
                    difference = round(float(my_sheet.cell(row, 12).value)) - round(float(my_sheet.cell(previous_row_non_dom , 13).value)) 
                    for diff in range(0,difference):
                        Y_non_dom.append(None)

                # Synchronisation des données des capteurs avec les annotations
                nb_repetitions = int(round(float(my_sheet.cell(label.row, 13).value))) - int(round(float(my_sheet.cell(label.row, 12).value)))
                for nb_sec in range(nb_repetitions): 
                    if X_non_dom and (len(Y_non_dom) == len(X_non_dom)): 
                        break
                    if label.value[3:13] == "sédentaire" :
                        Y_non_dom.append("non mouvement")
                    elif label.value[-2:] == "GA" or label.value[-2:] == "FA":
                        Y_non_dom.append("mouvement")
                    elif label.value[3:12] == "non noté":
                        Y_non_dom.append(None)

                #Au prochain tour, l'indice de la ligne actuelle sera l'index de la ligne précédente
                previous_row_non_dom = label.row

        # CROPER X A LA LONGUEUR DE Y 
        X_dom = X_dom[:len(Y_dom)] 
        X_non_dom = X_non_dom[:len(Y_non_dom)]
        Y_dom = Y_dom[:len(X_dom)]
        Y_non_dom = Y_non_dom[:len(X_non_dom)]

        all_X_dom.append(X_dom)
        all_X_non_dom.append(X_non_dom)
        all_Y_dom.append(Y_dom)
        all_Y_non_dom.append(Y_non_dom)

        # Mettre à jour les listes des longueurs de fichiers
        len_per_file_dom.append(len(X_dom) - previous_len_dom)
        len_per_file_non_dom.append(len(X_non_dom) - previous_len_non_dom)
        previous_len_dom = len(X_dom)
        previous_len_non_dom = len(X_non_dom)

print("Liste des longueurs pour dom : ", len_per_file_dom)
print("Liste des longueurs pour non dom : ", len_per_file_non_dom)

print("ATTENTION : le premier fichier n'est pas data 2 c'est data 10")


On est dans le fichier : Autres
On est dans le fichier : Data_10_dom.csv
Reading in CSV
Converting to array
Getting Counts
On est dans le fichier : Data_10_non_dom.csv
Reading in CSV
Converting to array
Getting Counts
On est dans le fichier : Data_10_X.xlsx
On est dans le fichier : Data_11_dom.csv
Reading in CSV
Converting to array
Getting Counts
On est dans le fichier : Data_11_non_dom.csv
Reading in CSV
Converting to array
Getting Counts
On est dans le fichier : Data_11_X.xlsx
On est dans le fichier : Data_2_dom.csv
Reading in CSV
Converting to array
Getting Counts
On est dans le fichier : Data_2_non_dom.csv
Reading in CSV
Converting to array
Getting Counts
On est dans le fichier : Data_2_X.xlsx
On est dans le fichier : Data_3_dom.csv
Reading in CSV
Converting to array
Getting Counts
On est dans le fichier : Data_3_non_dom.csv
Reading in CSV
Converting to array
Getting Counts
On est dans le fichier : Data_3_X.xlsx
On est dans le fichier : Data_4_dom.csv
Reading in CSV
Converting to a

In [9]:
print(len(all_X_dom[7]))
print(len(all_Y_dom[7]))
print(len(all_X_non_dom[7]))
print(len(all_Y_non_dom[7]))

2200
2200
2237
2237


Nettoyage des données

In [10]:
# Complexité : pour appliquer leave one out en divisant les fold par fichier (un fold = un fichier), il faut donner les X au Random Forest sous la forme [[fichier1], [fichier2]...], c'est à dire une liste de liste et non pas une simple liste comme c'est le cas dans les autres méthodes (autres fichiers python)

lst_X_members = [all_X_dom, all_X_non_dom]
lst_Y_members = [all_Y_dom, all_Y_non_dom]
# Ensembles pour tester le modèle
X_flat = [[],[]] # X aura la forme [[[fichier1],[fichier2]...],[[fichier1],[fichier2]...]] # avec [[membre dom],[membre non dom]]
Y_flat = [[],[]]

for idx_member in range(2):

    for idx_file in range(len(all_X_dom)): # all_X_dom et all_X_non_dom on la meme taille donnc peut import lequel on indique

        # ********************************** Préparation des données (équilibre, suppression des None,) *****************************************************************
        # Convertir en array pour faciliter le filtrage
        X_array = np.array(lst_X_members[idx_member][idx_file])
        Y_array = np.array(lst_Y_members[idx_member][idx_file])
        # Créer un masque pour garder uniquement les étiquettes valides (différent de None)
        mask_None = Y_array != None
        # Appliquer le masque
        X_clean = X_array[mask_None]
        Y_clean = Y_array[mask_None]

        # Aplatir les données (mettre au bon format pour les modèles)
        X_flat[idx_member].append(np.vstack(X_clean))
        Y_flat[idx_member].append(np.hstack(Y_clean))
        
        """# Création d'un mapping de couleur pour chaque label
        label_to_color = {'mouvement': 'red', 'non mouvement': 'blue'}
        colors = [label_to_color[y] for y in Y_flat[idx_member]]
        # Affichage : index en abscisse, intensité en ordonnée
        plt.scatter(range(len(X_flat[idx_member])), X_flat[idx_member][:, 0], c=colors, s=40)
        plt.title("Vérité non équilibrée pour seuil AC > 0")
        plt.xlabel("Index")
        plt.ylabel("Intensité")
        plt.show()
        print(len(X_array))
        print(Y_flat[idx_member])"""

# Ensembles pour entrainer et sauvegarder le modèle
X_array_dom_dump = np.array(X_dom)
Y_array_dom_dump = np.array(Y_dom)
X_array_non_dom_dump = np.array(X_non_dom)
Y_array_non_dom_dump = np.array(Y_non_dom)
# Créer un masque pour garder uniquement les étiquettes valides (différent de None)
mask_None_dom_dump = Y_array_dom_dump != None
mask_None_non_dom_dump = Y_array_non_dom_dump != None
# Appliquer le masque
X_clean_dom_dump = X_array_dom_dump[mask_None_dom_dump]
Y_clean_dom_dump = Y_array_dom_dump[mask_None_dom_dump]
X_clean_non_dom_dump = X_array_non_dom_dump[mask_None_non_dom_dump]
Y_clean_non_dom_dump = Y_array_non_dom_dump[mask_None_non_dom_dump]
# Aplatir les données (mettre au bon format pour les modèles)
X_flat_dom = np.vstack(X_clean_dom_dump)
Y_flat_dom = np.hstack(Y_clean_dom_dump)
X_flat_non_dom = np.vstack(X_clean_non_dom_dump)
Y_flat_non_dom = np.hstack(Y_clean_non_dom_dump)

In [11]:
print(len(X_flat[1][8]))
print(len(Y_flat[1][8]))

425
425


Apprentissage et enregistrement du modèle

In [12]:
brfc_dom = BalancedRandomForestClassifier()
brfc_non_dom = BalancedRandomForestClassifier()

brfc_dom.fit(X_flat_dom, Y_flat_dom)
brfc_non_dom.fit(X_flat_non_dom, Y_flat_non_dom)

joblib.dump(brfc_dom, 'RF_dom.joblib')
joblib.dump(brfc_non_dom, 'RF_non_dom.joblib')

['RF_non_dom.joblib']

Construction du modèle (avec leave-one-out et imbalanced class) pour le membre dominant

In [13]:
loo_dom = LeaveOneOut()
brfc_dom = BalancedRandomForestClassifier()

#print(X_flat[0])
scores_dom = []
for train_idx, test_idx in loo_dom.split(X_flat[0]): # X_flat[0] est l'ensemble des données du membre dominant : contient des listes conrrepondant aux fichiers

    # concaténer les fichiers d'entraînement
    X_train_dom = np.vstack([X_flat[0][i] for i in train_idx])
    Y_train_dom = np.hstack([Y_flat[0][i] for i in train_idx])
    # fichier de test
    X_test_dom = X_flat[0][test_idx[0]]
    Y_test_dom = np.array(Y_flat[0][test_idx[0]]).flatten()

    print(len(X_train_dom))

    brfc_dom.fit(X_train_dom, Y_train_dom)
    y_pred = brfc_dom.predict(X_test_dom)

    # F1 score
    #f1score_weighted_dom = f1_score(Y_test_dom, y_pred, labels = ["non mouvement", "mouvement"], average = "weighted")
    #print("F1 score mouvement : ", f1score_weighted_dom)

    # F1 score
    f1score_mouvement = f1_score(Y_test_dom, y_pred, labels = ["non mouvement","mouvement"], pos_label = "mouvement", average = "binary")
    f1score_non_mouvement = f1_score(Y_test_dom, y_pred, labels = ["non mouvement","mouvement"], pos_label = "non mouvement", average = "binary")
    f1score_weighted = f1_score(Y_test_dom, y_pred, labels = ["non mouvement","mouvement"], average = "weighted")
    f1score_macro = f1_score(Y_test_dom, y_pred, labels = ["non mouvement","mouvement"], average = "macro")

    """print("F1 score non mouvement: ", f1score_non_mouvement)
    print("F1 score mouvement : ", f1score_mouvement)
    print("F1 score weighted: ", f1score_weighted)
    print("F1 score macro : ", f1score_macro)"""

    scores_dom.append(f1score_weighted)

print(scores_dom)


10004
9116
10002
10747
11994
12005
11857
11877
11976
11689
[0.7489764279156612, 0.9315254150996274, 0.8412648750720616, 0.7456756011987389, 0.8449630720124495, 0.8818086401150206, 0.9673469387755103, 0.9113910585620164, 0.7436666752513157, 0.9111650578899707]


Construction du modèle (avec leave-one-out et imbalanced class) pour le membre non dominant

In [14]:
loo_non_dom = LeaveOneOut()
brfc_non_dom = BalancedRandomForestClassifier()
scores_non_dom_weighted = []
scores_non_dom_macro = []

for train_idx, test_idx in loo_non_dom.split(X_flat[1]): # X_flat[1] est l'ensemble des données du membre non dominant : contient des listes conrrepondant aux fichiers

    # concaténer les fichiers d'entraînement
    X_train_non_dom = np.vstack([X_flat[1][i] for i in train_idx])
    Y_train_non_dom = np.hstack([Y_flat[1][i] for i in train_idx])
    # fichier de test
    X_test_non_dom = X_flat[1][test_idx[0]]
    Y_test_non_dom = np.array(Y_flat[1][test_idx[0]]).flatten()

    print(len(X_test_non_dom))

    brfc_non_dom.fit(X_train_non_dom, Y_train_non_dom)
    y_pred = brfc_non_dom.predict(X_test_non_dom)


    # F1 score
    #f1score_weighted_dom = f1_score(Y_test_dom, y_pred, labels = ["non mouvement", "mouvement"], average = "weighted")
    #print("F1 score mouvement : ", f1score_weighted_dom)
    

    # F1 score
    f1score_mouvement = f1_score(Y_test_non_dom, y_pred, labels = ["non mouvement","mouvement"], pos_label = "mouvement", average = "binary")
    f1score_non_mouvement = f1_score(Y_test_non_dom, y_pred, labels = ["non mouvement","mouvement"], pos_label = "non mouvement", average = "binary")
    f1score_weighted = f1_score(Y_test_non_dom, y_pred, labels = ["non mouvement","mouvement"], average = "weighted")
    f1score_macro = f1_score(Y_test_non_dom, y_pred, labels = ["non mouvement","mouvement"], average = "macro")

    print("F1 score non mouvement: ", f1score_non_mouvement)
    print("F1 score mouvement : ", f1score_mouvement)
    print("F1 score weighted: ", f1score_weighted)
    print("F1 score macro : ", f1score_macro)

    scores_non_dom_weighted.append(f1score_weighted)
    scores_non_dom_macro.append(f1score_macro)

scores_non_dom_weighted = np.array(scores_non_dom_weighted)
print(np.mean(scores_non_dom_weighted))
scores_non_dom_macro = np.array(scores_non_dom_macro)
print(np.mean(scores_non_dom_macro))


2434
F1 score non mouvement:  0.3247863247863248
F1 score mouvement :  0.6193382589142307
F1 score weighted:  0.5826705366296147
F1 score macro :  0.4720622918502777
3235
F1 score non mouvement:  0.09750297265160524
F1 score mouvement :  0.8651625510747912
F1 score weighted:  0.849263573716413
F1 score macro :  0.48133276186319823
2711
F1 score non mouvement:  0.49886963074604374
F1 score mouvement :  0.8376068376068376
F1 score weighted:  0.7830040491899557
F1 score macro :  0.6682382341764407
1618
F1 score non mouvement:  0.6935483870967742
F1 score mouvement :  0.7391304347826086
F1 score weighted:  0.7238894658097802
F1 score macro :  0.7163394109396914
382
F1 score non mouvement:  0.6111111111111112
F1 score mouvement :  0.9595375722543352
F1 score weighted:  0.9221410149065022
F1 score macro :  0.7853243416827231
292
F1 score non mouvement:  0.79
F1 score mouvement :  0.890625
F1 score weighted:  0.8582320205479452
F1 score macro :  0.8403125
504
F1 score non mouvement:  0.0
F1 s